# Design of Buffer Tanks

This notebook implements the results and recommendations from the following paper:

- Faanes and Skogestad (2000). A systematic approach to the design of buffer tanks

## Model

Consider a buffer tank with liquid volume $V$, inlet flow rate
$q_{\text{in}}$, and outlet flow rate $q$.

The inlet and outlet quality variables (e.g. concentration or temperature) are
denoted by $c_{\text{in}}$ and $c$, respectively.

For a perfectly mixed tank, a component (or simplified energy) balance yields

$$
\frac{d(Vc)}{dt} = q_{\text{in}}\,c_{\text{in}} - q\,c
$$

In addition, the total mass balance (assuming constant density) is

$$
\frac{dV}{dt} = q_{\text{in}} - q
$$

**Symbols**

- $V$ — liquid volume in the tank $[ \mathrm{m}^3 ]$
- $q_{\text{in}}$ — inlet volumetric flow rate $[ \mathrm{m}^3\,\mathrm{s}^{-1} ]$
- $q$ — outlet volumetric flow rate $[ \mathrm{m}^3\,\mathrm{s}^{-1} ]$
- $c_{\text{in}}$ — inlet flow quality (concentration or temperature)  
  $[ \mathrm{kg}\,\mathrm{m}^{-3} \text{ or } \mathrm{K} ]$
- $c$ — outlet flow quality  
  $[ \mathrm{kg}\,\mathrm{m}^{-3} \text{ or } \mathrm{K} ]$
- $t$ — time $[ \mathrm{s} ]$


## Smoothing Quality Variations

### Transfer function for a single buffer tank

For quality disturbances, the objective of the buffer tank is to smoothen the quality response, $c(s) = h(s) q_{in}(s)$, so that the variations in $c$ are smaller than in $c_{in}$.

By linearizing at the steady-state operating point we get the following function for the quality of the outlet flow.

$$
c(s)=\frac{1}{\frac{V^*}{q^*} s+1}\left[c_{\text {in }}(s)+\frac{c^*_{\text {in }}-c^*}{q^*} q_{\text {in }}(s)\right]
$$

where $^*$ denotes the nominal (steady state) values.

In the case where $c^*_{\text {in}} = c^*$, the transfer function reduces to

$$
h(s) = \frac{1}{\tau_h s + 1}
$$

where $\tau_h = V^* / q^*$ is called the residence time (steady state).

### Transfer function for buffer tanks in series

$$
h(s) = \frac{1}{\big( \frac{\tau_h}{n} s + 1 \big)^n}
$$

## Frequency Response


**Magnitude**

$$
\left| H(j\omega) \right|
=
\frac{1}{
\left[
1 + \left( \omega \frac{\tau_h}{n} \right)^2
\right]^{\tfrac{n}{2}}
}
$$

**Phase**

$$
\angle H(j\omega)
=
-\,n \tan^{-1}\!\left(
\omega \frac{\tau_h}{n}
\right)
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def calc_amplitude_response(omega, tau_h, n):
    """Calculate the amplitude response of n buffer tanks in series.
    """
    return 1 / (1 + (omega * tau_h / n)**2) ** (n / 2)


def calc_phase_response(omega, tau_h, n):
    """Calculate the phase response of n buffer tanks in series.
    """
    return -n * np.arctan(omega * tau_h)

In [ ]:
# Input disturbance parameters
period = 10  # period of inlet disturbance (min)
omega = 2 * np.pi / period  # angular frequency (rad/min)
freq_times_tau_h = np.logspace(-1, 2, 101)  # total volume of buffer tanks (m^3)

amplitude_responses = {}
for n_tanks in [1, 2, 3, 4]:  # number of tanks in series
    tau_h = freq_times_tau_h / omega  # residence time in a single tank (min)
    amplitude_responses[n_tanks] = calc_amplitude_response(omega, tau_h, n_tanks)

plt.figure(figsize=(7, 5.5))
for n_tanks, amplitude_response in amplitude_responses.items():
    plt.loglog(freq_times_tau_h, amplitude_response, label=n_tanks)

plt.xlim([1e-1, 1e2])
plt.ylim([1e-3, 1.2])
plt.xlabel(r'Frequency $\times$ residence time')
plt.ylabel('Amplitude Response')
plt.grid()
#plt.legend(title='$n$')
plt.annotate('n=1', xy=(48, 0.03), color='C0')
plt.annotate('n=2', xy=(29, 0.007), color='C1')
plt.annotate('n=3', xy=(20, 0.0035), color='C2')
plt.annotate('n=4', xy=(11, 0.002), color='C3')
plt.title('Frequency Responses of Buffer Tanks in Series')
plt.show()

In [ ]:
# Parameters
q_in = 1 # inlet flow-rate m^3/min

# Input disturbance parameters
period = 10  # period of inlet disturbance (min)
omega = 2 * np.pi / period  # angular frequency (rad/min)
total_volume = np.logspace(0, 3, 101)  # total volume of buffer tanks (m^3)

amplitude_responses = {}
for n_tanks in [1, 2, 3, 4]:  # number of tanks in series
    tau_h = freq_times_tau_h / omega  # residence time in a single tank (min)

    amplitude_responses[n_tanks] = calc_amplitude_response(omega, tau_h, n_tanks)

plt.figure(figsize=(7, 5.5))
for n_tanks, amplitude_response in amplitude_responses.items():
    plt.loglog(total_volume, amplitude_response, label=n_tanks)

plt.xlim([1, 1e3])
plt.ylim([1e-3, 1.2])
plt.xlabel(r'Total tank volume (m$^3$)')
plt.ylabel('Amplitude Response')
plt.grid()
#plt.legend(title='$n$')
plt.annotate('n=1', xy=(480, 0.03), color='C0')
plt.annotate('n=2', xy=(290, 0.007), color='C1')
plt.annotate('n=3', xy=(200, 0.0035), color='C2')
plt.annotate('n=4', xy=(110, 0.002), color='C3')
plt.title('Frequency Responses of Buffer Tanks in Series')
plt.show()